# K6 · Đánh giá Agent (T12): kịch bản đặt bàn nhiều lượt + định tuyến công cụ
Kịch bản: `data/splits/scenarios.jsonl` (viết tay). Nếu chưa có → dùng kịch bản **proxy** dựng từ val v2 (chỉ kiểm tra pipeline, không phải số chính thức). Công cụ chạy trên bản sao DB. Định tuyến công cụ: đọc file `gen/*_agent_*.jsonl` của K5 (Add Data: output K5) hoặc tự chạy trên test_synth.

In [ ]:
%%capture
!pip install -q "transformers==4.56.2" "trl==0.21.0" "peft==0.17.0" "bitsandbytes==0.47.0" "accelerate>=1.4" "datasets>=3.0" rank-bm25 optuna

In [ ]:
import sys, glob, json, os, time
sys.path.insert(0, glob.glob('/kaggle/input/**/code/src', recursive=True)[0])
from kimochi.kaggle_env import setup, gpu_info
ROOT = setup()
import pandas as pd
from pathlib import Path
RES = ROOT / 'results'
print(gpu_info())

In [ ]:
from kimochi import llm, agent_eval as E, prompts, retriever, agent
import shutil
ADAPTER = None   # None = tự tìm adapter tốt nhất trong output K4
MODEL, QUANT = 'qwen', 'nf4'
if ADAPTER is None:
    found = sorted(glob.glob('/kaggle/input/**/runs/*_s42/adapter', recursive=True))
    ADAPTER = found[0] if found else None
    if ADAPTER: MODEL, QUANT = Path(ADAPTER).parent.name.split('_')[:2]
print(MODEL, QUANT, ADAPTER)
model, tok, _ = llm.load(MODEL, QUANT, adapter=ADAPTER)
gen = lambda msgs: llm.generate(model, tok, [msgs], bs=1)[0][0]

## Kịch bản nhiều lượt

In [ ]:
sc_path = ROOT / 'data/splits/scenarios.jsonl'
if sc_path.exists():
    scs = [json.loads(l) for l in sc_path.read_text(encoding='utf-8').splitlines() if l.strip()]; kind = 'viết tay'
else:
    scs = E.scenarios_from_v2_val(ROOT); kind = 'PROXY (val v2)'
print(len(scs), 'kịch bản', kind)
W = Path('/kaggle/working/sc'); W.mkdir(exist_ok=True)
res = pd.DataFrame([E.run_scenario(sc, gen, ROOT / 'data/kimochi.db', W) for sc in scs])
res['kind'] = kind; res.to_json(RES / 'agent_scenarios.jsonl', orient='records', lines=True, force_ascii=False)
summary = {'kind': kind, 'n': len(res), 'task_success': res.success.mean(), 'db_state_ok': res.db_state_ok.mean(),
           'hitl_ok': res.hitl_ok.mean(), 'avg_tool_calls': res.n_tool_calls.mean()}
(RES / 'agent_summary.json').write_text(json.dumps(summary, indent=1, default=float)); summary

In [ ]:
res[~res.success][['id', 'db_state_ok', 'hitl_ok', 'tools', 'answers']].head(10)  # phân tích lỗi

## Định tuyến công cụ (confusion matrix + chi phí kinh doanh)

In [ ]:
files = glob.glob('/kaggle/input/**/gen/*_agent_test_*.jsonl', recursive=True)
if files:
    ag = pd.read_json(sorted(files)[0], lines=True); src = Path(sorted(files)[0]).name
else:
    ev = prompts.load_eval_set('test_synth')
    R = retriever.HybridRetriever(retriever.load_docs())
    db = Path('/kaggle/working/route.db'); shutil.copy(ROOT / 'data/kimochi.db', db)
    out = agent.run_batch(model, tok, list(ev['query']), db, contexts=[R.search_chunks(q, 'bm25_filter', k=3) for q in ev['query']])
    ag = ev.assign(tools_called=[o['tools_called'] for o in out]); src = 'test_synth (chạy trực tiếp)'
rep = E.routing_report(ag)
print(src, '| accuracy', round(rep['accuracy'], 3), '| macro-F1', round(rep['macro_f1'], 3), '| chi phí / 100 câu', round(rep['cost_per_100'], 1))
rep['confusion'].to_csv(RES / 'tool_routing_confusion.csv'); rep['per_label'].to_csv(RES / 'tool_routing_per_label.csv')
(RES / 'tool_routing_summary.json').write_text(json.dumps({k: rep[k] for k in ('accuracy', 'macro_f1', 'business_cost', 'cost_per_100')} | {'source': src}, indent=1))
E.plot_confusion(rep['confusion'], RES / 'tool_routing_confusion.png'); display(rep['confusion']); rep['per_label'].round(3)